# nnU-Net v2 on MSD Task06 Lung, OncoNova split (Colab)

OncoNova AI is an educational research prototype, not a clinically validated diagnostic system.

This notebook trains **one nnU-Net v2 `3d_fullres` model** as a Stage 3 candidate, on exactly the patients OncoNova's own candidates use:

| Patients | Count | Used here for |
|---|---|---|
| OncoNova *fit* (from the manifest's train split) | 37 | training |
| OncoNova *inner validation* (from the train split) | 7 | nnU-Net's fold-0 validation (monitoring) |
| Manifest *val* | 9 | **not copied**: OncoNova compares models on these, locally |
| Manifest *test* | 10 | **not copied**: locked until the final evaluation |

The 19 val and test patients never enter nnU-Net's raw dataset, so they are not seen by its fingerprinting, planning, training or checkpoint choice.

**Data:** MSD Task06 Lung, CC BY-SA 4.0 (Antonelli et al., Nat Commun 2022). It is downloaded here from the MONAI mirror on AWS Open Data and checked against MONAI's MD5.

**Runtime:** this needs a CUDA GPU (Runtime → Change runtime type → GPU). `nnUNetTrainer_100epochs` is the default. I have not timed it on Colab, so expect several hours on a T4 and check the first epochs' times. If the session may time out, set `USE_DRIVE = True` so the results survive and training can resume with `--c`.

**Output:** a zip containing `checkpoint_final.pth` and the plans. Put it in `artifacts/models/nnunet-msd/` in the OncoNova repo; it is evaluated there on the 9 val patients with the same protocol as every other candidate.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU first."
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install nnunetv2==2.8.1
import nnunetv2, importlib.metadata as md
print("nnunetv2", md.version("nnunetv2"))

In [ ]:
import os, json, shutil, hashlib, tarfile, urllib.request
from pathlib import Path

USE_DRIVE = False          # True: keep nnUNet_results on Google Drive so a dropped session can resume
TRAINER = "nnUNetTrainer_100epochs"   # nnUNetTrainer_250epochs, or nnUNetTrainer (1000 epochs) on a fast GPU
DATASET_ID, DATASET_NAME = 601, "Dataset601_OncoNovaLung"

base = Path("/content/nnunet")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    results = Path("/content/drive/MyDrive/onconova_nnunet_results")
else:
    results = base / "results"
for name, p in {"nnUNet_raw": base / "raw", "nnUNet_preprocessed": base / "preprocessed", "nnUNet_results": results}.items():
    p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
print({k: os.environ[k] for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results")})

In [ ]:
# Download and verify MSD Task06 Lung (9.16 GB), then extract with path checks.
URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"
MD5 = "8afd997733c7fc0432f71255ba4e52dc"   # monai.apps.datasets.DecathlonDataset
tar_path = Path("/content/Task06_Lung.tar")
if not tar_path.exists():
    urllib.request.urlretrieve(URL, tar_path)
h = hashlib.md5()
with tar_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == MD5, f"MD5 mismatch: {h.hexdigest()}"
with tarfile.open(tar_path) as t:
    t.extractall("/content", filter="data")
task = Path("/content/Task06_Lung")
print(sorted(os.listdir(task)))

In [ ]:
# OncoNova's split (from data/external/msd-task06-lung/manifest.json and configs/segmentation_unet_msd.yaml).
FIT = ["lung_005", "lung_006", "lung_010", "lung_014", "lung_018", "lung_020", "lung_026", "lung_027", "lung_028", "lung_033", "lung_034", "lung_037", "lung_038", "lung_041", "lung_042", "lung_043", "lung_044", "lung_045", "lung_047", "lung_048", "lung_051", "lung_055", "lung_057", "lung_058", "lung_059", "lung_062", "lung_065", "lung_071", "lung_074", "lung_078", "lung_080", "lung_081", "lung_083", "lung_084", "lung_086", "lung_093", "lung_096"]
INNER_VAL = ["lung_009", "lung_016", "lung_029", "lung_046", "lung_069", "lung_075", "lung_092"]
NOT_COPIED_VAL = ["lung_001", "lung_022", "lung_031", "lung_036", "lung_053", "lung_061", "lung_066", "lung_070", "lung_073"]  # manifest val: compared locally; manifest test (10) is not listed either
assert len(FIT) == 37 and len(INNER_VAL) == 7 and not set(FIT) & set(INNER_VAL) and not (set(FIT) | set(INNER_VAL)) & set(NOT_COPIED_VAL)

raw = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
(raw / "imagesTr").mkdir(parents=True, exist_ok=True)
(raw / "labelsTr").mkdir(parents=True, exist_ok=True)
for case in FIT + INNER_VAL:
    shutil.copy(task / "imagesTr" / f"{case}.nii.gz", raw / "imagesTr" / f"{case}_0000.nii.gz")
    shutil.copy(task / "labelsTr" / f"{case}.nii.gz", raw / "labelsTr" / f"{case}.nii.gz")
(raw / "dataset.json").write_text(json.dumps({
    "channel_names": {"0": "CT"}, "labels": {"background": 0, "cancer": 1},
    "numTraining": len(FIT) + len(INNER_VAL), "file_ending": ".nii.gz",
    "licence": "CC BY-SA 4.0 (MSD Task06 Lung)", "description": "OncoNova split: 37 fit + 7 inner-val patients only"}, indent=2))
print(len(os.listdir(raw / "imagesTr")), "images copied")

In [ ]:
!nnUNetv2_plan_and_preprocess -d 601 -c 3d_fullres --verify_dataset_integrity

In [ ]:
# One fold: train on the 37 fit patients, validate on the 7 inner-val patients.
pre = Path(os.environ["nnUNet_preprocessed"]) / DATASET_NAME
(pre / "splits_final.json").write_text(json.dumps([{"train": FIT, "val": INNER_VAL}], indent=2))
print(json.loads((pre / "splits_final.json").read_text())[0].keys())

In [ ]:
# Train fold 0. If the session drops (with USE_DRIVE = True), rerun this cell with --c added to continue.
!nnUNetv2_train 601 3d_fullres 0 -tr {TRAINER}

In [ ]:
# Package what OncoNova needs: the final checkpoint, plans and logs (no images).
out_dir = Path(os.environ["nnUNet_results"]) / DATASET_NAME / f"{TRAINER}__nnUNetPlans__3d_fullres"
pkg = Path("/content/nnunet-msd")
if pkg.exists():
    shutil.rmtree(pkg)
(pkg / "fold_0").mkdir(parents=True)
for f in ("plans.json", "dataset.json", "dataset_fingerprint.json"):
    shutil.copy(out_dir / f, pkg / f)
for f in os.listdir(out_dir / "fold_0"):
    if f.endswith((".pth", ".json", ".png", ".txt")) and f != "checkpoint_latest.pth":
        shutil.copy(out_dir / "fold_0" / f, pkg / "fold_0" / f)
(pkg / "onconova_split.json").write_text(json.dumps({"fit": FIT, "inner_val": INNER_VAL, "trainer": TRAINER,
                                                      "nnunetv2": md.version("nnunetv2"), "gpu": torch.cuda.get_device_name(0)}, indent=2))
shutil.make_archive("/content/nnunet-msd", "zip", pkg)
print(sorted(p.name for p in pkg.rglob("*")))
from google.colab import files
files.download("/content/nnunet-msd.zip")

## Back in OncoNova

1. Unzip into `artifacts/models/nnunet-msd/` (it holds `plans.json`, `dataset.json`, `fold_0/checkpoint_final.pth`, ...). That folder is gitignored.
2. Ask Claude to evaluate it. It runs on the 9 val patients with `scripts/evaluate_segmentation.py`, with the same metrics, latency and memory as the other candidates, and is added to the comparison table.

`checkpoint_final.pth` is used, not `checkpoint_best.pth`, matching nnU-Net's default for inference.